# Startup Idea Validator — Deep Research Agent

A multi-agent pipeline that takes a one-sentence startup idea, researches the market using web search, and produces a structured viability verdict.

**Agent pipeline:**
1. **Planner Agent** — generates 5 targeted search queries (market size, competitors, funding, demand, barriers)
2. **Search Agents** — run in parallel using `WebSearchTool`
3. **Analyst Agent** — synthesizes research into a structured `StartupValidation`
4. **Writer Agent** — produces a polished markdown report

Optional delivery via **Pushover** notifications and **SendGrid** email.

In [1]:
from agents import Agent, WebSearchTool, Runner, trace
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from IPython.display import display, Markdown
import asyncio
import requests
import os

load_dotenv(override=True)

True

## Pydantic Models (Structured Outputs)

We define four schemas:
- **WebSearchItem / WebSearchPlan** — the planner's output, directing what to search
- **StartupValidation** — the analyst's structured verdict (the core value)
- **ReportData** — the writer's final deliverable

In [2]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search angle matters for validating the idea.")
    query: str = Field(description="The search term to use.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="Exactly 5 web searches covering market size, competitors, funding, demand signals, and barriers."
    )


class StartupValidation(BaseModel):
    verdict: str = Field(
        description="One of: 'High Potential', 'Proceed with Caution', or 'Crowded Market'."
    )
    confidence: int = Field(description="Confidence in the verdict from 1 (low) to 10 (high).")
    market_size: str = Field(description="Estimated TAM, e.g. '$4.2B by 2027'.")
    top_competitors: list[str] = Field(description="Names of the most relevant existing players.")
    similar_funded_startups: list[str] = Field(
        description="Recently funded startups tackling a similar problem."
    )
    key_risks: list[str] = Field(description="The biggest risks this idea faces.")
    key_opportunities: list[str] = Field(description="The strongest opportunities or tailwinds.")
    what_would_need_to_be_true: list[str] = Field(
        description="VC-style beliefs that must hold for this idea to succeed."
    )
    recommendation: str = Field(description="A concise final recommendation paragraph.")


class ReportData(BaseModel):
    short_summary: str = Field(description="A 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The full report in markdown.")
    follow_up_questions: list[str] = Field(description="Suggested areas to research further.")

## Define the Agents

Each agent has a single responsibility:
- **Planner** — turns an idea into 5 search queries
- **Search** — executes one web search and summarizes results
- **Analyst** — produces the structured `StartupValidation` from research
- **Writer** — turns the validation into a detailed markdown report

In [3]:
NUM_SEARCHES = 5

planner_agent = Agent(
    name="PlannerAgent",
    instructions=(
        "You are a startup validation strategist. Given a startup idea, generate exactly "
        f"{NUM_SEARCHES} web search queries to validate it. Each query must target a distinct "
        "angle:\n"
        "1. Market size / TAM for the problem space\n"
        "2. Existing competitors and alternatives\n"
        "3. Recent funding rounds or acquisitions in the space\n"
        "4. Customer pain points, demand signals, or forum discussions\n"
        "5. Regulatory, technical, or adoption barriers\n\n"
        "Make queries specific and current (include the current year when relevant)."
    ),
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

search_agent = Agent(
    name="SearchAgent",
    instructions=(
        "You are a research assistant. Given a search term, search the web and produce a "
        "concise summary of the results. The summary must be 2-3 paragraphs and under 300 "
        "words. Capture concrete data points — numbers, company names, dates, dollar amounts. "
        "Write succinctly; this will be consumed by an analyst, so capture the essence and "
        "skip the fluff. Do not add commentary beyond the summary."
    ),
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

analyst_agent = Agent(
    name="AnalystAgent",
    instructions=(
        "You are a startup analyst at a top-tier VC firm. Given the original startup idea and "
        "a set of research summaries, produce a structured viability assessment.\n\n"
        "Rules:\n"
        "- Be brutally honest. Founders deserve the truth.\n"
        "- Ground every claim in the research provided — do not invent data.\n"
        "- If data is missing or inconclusive, say so and lower your confidence score.\n"
        "- The 'what_would_need_to_be_true' field is critical: list the assumptions that must "
        "hold for this startup to reach product-market fit and scale.\n"
        "- Verdict must be exactly one of: 'High Potential', 'Proceed with Caution', "
        "or 'Crowded Market'."
    ),
    model="gpt-4o-mini",
    output_type=StartupValidation,
)

writer_agent = Agent(
    name="WriterAgent",
    instructions=(
        "You are a senior analyst writing a polished startup validation report. You will "
        "receive the original idea and a structured validation object.\n\n"
        "Write a detailed markdown report with these sections:\n"
        "1. **Executive Summary** — verdict, confidence, and one-paragraph recommendation\n"
        "2. **Market Analysis** — TAM, growth drivers, timing\n"
        "3. **Competitive Landscape** — key players, their strengths, white space\n"
        "4. **Funding Activity** — recent rounds, acqui-hires, signals\n"
        "5. **Risks & Opportunities** — side by side\n"
        "6. **What Would Need to Be True** — the beliefs required for success\n"
        "7. **Recommendation & Next Steps**\n\n"
        "Aim for at least 800 words. Use concrete data from the validation object."
    ),
    model="gpt-4o-mini",
    output_type=ReportData,
)

## Orchestration Functions

These functions wire the agents together:
1. `plan_searches` — asks the planner for 5 queries
2. `search` / `perform_searches` — runs all 5 searches in parallel
3. `analyze` — feeds research into the analyst
4. `write_report` — produces the final polished report

In [4]:
async def plan_searches(idea: str) -> WebSearchPlan:
    result = await Runner.run(planner_agent, f"Startup idea: {idea}")
    return result.final_output


async def search(item: WebSearchItem) -> str:
    input_text = f"Search term: {item.query}\nReason: {item.reason}"
    result = await Runner.run(search_agent, input_text)
    return result.final_output


async def perform_searches(plan: WebSearchPlan) -> list[str]:
    tasks = [asyncio.create_task(search(item)) for item in plan.searches]
    return await asyncio.gather(*tasks)


async def analyze(idea: str, search_results: list[str]) -> StartupValidation:
    input_text = (
        f"Startup idea: {idea}\n\n"
        f"Research summaries:\n" + "\n---\n".join(search_results)
    )
    result = await Runner.run(analyst_agent, input_text)
    return result.final_output


async def write_report(idea: str, validation: StartupValidation) -> ReportData:
    input_text = (
        f"Startup idea: {idea}\n\n"
        f"Structured validation data:\n{validation.model_dump_json(indent=2)}"
    )
    result = await Runner.run(writer_agent, input_text)
    return result.final_output

## Optional Delivery — Pushover Notifications

Sends a short push notification with the verdict and confidence score. Only runs if `PUSHOVER_TOKEN` is set in your `.env`.

In [5]:
def push(text: str):
    """Send a Pushover notification."""
    requests.post(
        "https://api.pushover.net/1/messages.json",
        data={
            "token": os.getenv("PUSHOVER_TOKEN"),
            "user": os.getenv("PUSHOVER_USER"),
            "message": text,
        },
    )


def deliver_pushover(validation: StartupValidation):
    msg = (
        f"Startup Validator Result\n"
        f"Verdict: {validation.verdict} ({validation.confidence}/10)\n"
        f"Market: {validation.market_size}\n"
        f"Recommendation: {validation.recommendation[:200]}"
    )
    push(msg)

## Run It!

Enter your startup idea below. The full pipeline will:
1. Plan 5 targeted searches
2. Execute them in parallel
3. Analyze the results into a structured verdict
4. Write a detailed report
5. Send a Pushover notification (if configured)

**Note:** Each web search costs ~2.5 cents. With 5 searches, expect ~12-15 cents per run.

In [6]:
IDEA = "An AI-powered meal planning app for people with dietary restrictions"

with trace("Startup Validation"):
    print("Step 1: Planning searches...")
    search_plan = await plan_searches(IDEA)
    for i, s in enumerate(search_plan.searches, 1):
        print(f"  {i}. {s.query}  ({s.reason})")

    print("\nStep 2: Running searches in parallel...")
    search_results = await perform_searches(search_plan)
    print(f"  Got {len(search_results)} results")

    print("\nStep 3: Analyzing...")
    validation = await analyze(IDEA, search_results)

    print("\nStep 4: Writing report...")
    report = await write_report(IDEA, validation)

    if os.getenv("PUSHOVER_TOKEN"):
        print("\nStep 5: Sending Pushover notification...")
        deliver_pushover(validation)

    print("\nDone!")

Step 1: Planning searches...
  1. 2023 market size for meal planning apps catering to dietary restrictions  (Understanding the potential market size will help gauge the viability of the startup.)
  2. 2023 existing competitors in AI meal planning apps for dietary restrictions  (Identifying competitors and their offerings will help differentiate the startup's features and positioning.)
  3. 2023 funding rounds for meal planning tech startups focusing on dietary needs  (Recent funding or acquisitions can indicate investor interest and market dynamics.)
  4. 2023 customer reviews and forum discussions on meal planning for dietary restrictions  (Examining customer pain points can validate the demand for such a product and its features.)
  5. 2023 regulatory barriers and adoption challenges for meal planning apps in the health tech space  (Identifying barriers will help in understanding what challenges need to be addressed for successful implementation.)

Step 2: Running searches in paralle

[non-fatal] Tracing: request failed: timed out


  Got 5 results

Step 3: Analyzing...

Step 4: Writing report...

Step 5: Sending Pushover notification...

Done!


## View the Structured Verdict

The raw `StartupValidation` object — this is what makes this agent different from a generic research agent.

In [7]:
print(f"VERDICT:    {validation.verdict}")
print(f"CONFIDENCE: {validation.confidence}/10")
print(f"MARKET:     {validation.market_size}")
print(f"\nTOP COMPETITORS: {', '.join(validation.top_competitors)}")
print(f"\nRECENTLY FUNDED: {', '.join(validation.similar_funded_startups)}")
print(f"\nKEY RISKS:")
for r in validation.key_risks:
    print(f"  - {r}")
print(f"\nKEY OPPORTUNITIES:")
for o in validation.key_opportunities:
    print(f"  + {o}")
print(f"\nWHAT WOULD NEED TO BE TRUE:")
for b in validation.what_would_need_to_be_true:
    print(f"  ? {b}")
print(f"\nRECOMMENDATION:\n{validation.recommendation}")

VERDICT:    Proceed with Caution
CONFIDENCE: 6/10
MARKET:     $704.7 million by 2030

TOP COMPETITORS: Vitalmint, NourishMate, MealPlanner, Bite AI, Tera, What's For Dinner, Hungryroot

RECENTLY FUNDED: PlateJoy, EatWell, NutriPlan

KEY RISKS:
  - High competition from established AI meal planning apps focused on dietary needs
  - User concerns about accuracy of allergen information
  - Data privacy and regulatory compliance issues
  - Potential high churn rates due to complexity or lack of personalization

KEY OPPORTUNITIES:
  + Growing demand for personalized meal planning solutions
  + Increasing health consciousness and focus on dietary restrictions
  + Recent funding trends indicate strong investor interest in this sector
  + Integration possibilities with fitness and health apps

WHAT WOULD NEED TO BE TRUE:
  ? The app must reliably identify and avoid cross-contamination for dietary restrictions
  ? User engagement features must promote long-term use to overcome high churn rates


## Full Report

The writer agent's detailed markdown report, ready to share or present.

In [8]:
display(Markdown(report.markdown_report))

# Startup Validation Report: AI-Powered Meal Planning App

## Executive Summary
The idea of an AI-powered meal planning app for individuals with dietary restrictions receives a verdict of **Proceed with Caution**. With a confidence level of **6**, it is evident that while the market shows potential, challenges including intense competition, user concerns, and regulatory compliance pose significant risks. I recommend that founders prioritize building robust allergen identification features and ensure compliance with health regulations to effectively differentiate their product. 

## Market Analysis
The meal planning app market is projected to grow substantially, with an estimated Total Addressable Market (TAM) of **$704.7 million by 2030**. Growth drivers include an increasing societal focus on health and nutrition, leading to higher demand for personalized meal solutions that cater not just to dietary preferences but also to restrictions (e.g., allergens, health conditions). Timing is pivotal, as consumer habits are shifting towards health-conscious decision-making, especially post-pandemic, amplifying the need for such tailored solutions.  

## Competitive Landscape
The competitive landscape is quite crowded, with several key players already established in the market:
- **Vitalmint**
- **NourishMate**
- **MealPlanner**
- **Bite AI**
- **Tera**
- **What's For Dinner**
- **Hungryroot**

These competitors have strengths in user engagement, established customer bases, and seamless app functionalities. However, the significant **white space** exists in niche markets targeting unique dietary restrictions, indicating an opportunity for specialization that caters to specific individual needs. Emphasizing features around allergen safety and meal adaptability could set a new entrant apart. 

## Funding Activity
Recent funding activity shows promising interest within this sector, as evidenced by similar funded startups like **PlateJoy**, **EatWell**, and **NutriPlan**. This indicates not only high investor interest but also a validation of the business model potential. Recent rounds have underscored an eagerness from investors to back innovations in the personalized health and meal planning space. It’s imperative, however, to weigh this interest against the competitive risk factors involved. Acqui-hires, although not directly observed for this vertical, have occurred in adjacent markets, suggesting a trend where established companies may seek to integrate innovative solutions rapidly.  

## Risks & Opportunities
| **Risks** | **Opportunities** |
|------------|-------------------|
| High competition from established AI meal planning apps focused on dietary needs. | Growing demand for personalized meal planning solutions. |
| User concerns about accuracy of allergen information could lead to distrust. | Increasing health consciousness and focus on dietary restrictions enhances market relevance. |
| Data privacy and regulatory compliance issues may complicate development. | Recent funding trends indicate strong investor interest in this sector. |
| Potential high churn rates due to complexity or lack of personalization could affect retention. | Opportunities for integration with fitness and health apps could drive user engagement. |  

## What Would Need to Be True
For the success of the proposed app, several foundational beliefs must be validated:
- The app must reliably identify and avoid cross-contamination associated with dietary restrictions.  
- User engagement features must be built into the app to promote long-term usage and decrease churn over time.  
- Regulatory compliance is non-negotiable to avoid penalties and maintain user trust.  
- The app should seamlessly integrate with existing health systems and align with user routines to enhance usability.  

## Recommendation & Next Steps
Given the cautious approach advised in this validation, it is essential for stakeholders to consider the following next steps:
1. **Market Research**: Conduct further qualitative research to understand consumer pain points regarding allergen identification and meal planning efficiency.  
2. **Prototyping**: Develop a minimum viable product (MVP) that tests key functionalities, particularly allergen detection and integration features.  
3. **Pilot Testing**: Engage in pilot programs with targeted user groups to collect feedback on app usability, features, and dietary satisfaction.  
4. **Compliance Review**: Consult with legal experts to ensure product development adheres to health regulations, especially concerning food safety and user data protection.  
5. **Investor Outreach**: Begin discussions with potential investors to secure funding for further development based on validated features and market needs.  

Addressing these specific areas lays a critical foundation for turning the startup idea into a viable and competitive offering in the meal planning app marketplace.

## Follow-Up Questions

The writer agent suggests areas to dig deeper into.

In [ ]:
for i, q in enumerate(report.follow_up_questions, 1):
    print(f"{i}. {q}")

## Check the Trace

View the full agent execution trace on the OpenAI dashboard:

https://platform.openai.com/traces